# DeepLure AIE-CASE: Color-Invariant Saree Design Recognition
### End-to-End Metric Learning Pipeline (Kaggle / Colab GPU Ready)

**Objective**: Build and evaluate a system that identifies a saree by the surface design/motif, independent of the colorway palette.
- **Architecture**: `ConvNeXt-Tiny` backbone + 512-D $L_2$-normalized embedding projection head
- **Loss Function**: `ArcFace (Additive Angular Margin Loss)` ($m=0.50, s=30.0$)
- **Color Invariance**: Grayscale dropout, extreme hue/saturation jitter, solarization, spatial augmentations
- **Dual Protocol Evaluation**: 1:N Identification (Rank-1, Rank-5, mAP) & 1:1 Verification (ROC-AUC, EER, Optimal Threshold)

In [ ]:
# 1. Dependencies and Environment Setup
!pip install -q timm scikit-learn

import math, os, random, time
from pathlib import Path
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional, Union

import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using compute device: {device}")

In [ ]:
# 2. Configuration
@dataclass
class Config:
    seed: int = 42
    image_size: int = 224
    backbone: str = 'convnext_tiny'
    embedding_dim: int = 512
    arcface_margin: float = 0.50
    arcface_scale: float = 30.0
    epochs: int = 10
    batch_size: int = 32
    learning_rate: float = 3e-4
    weight_decay: float = 1e-4
    grayscale_prob: float = 0.35
    solarize_prob: float = 0.15
    hue_jitter: float = 0.4
    sat_jitter: float = 0.4

cfg = Config()
torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)
random.seed(cfg.seed)

In [ ]:
# 3. Color-Invariance Data Augmentations
def get_transforms(image_size=224, is_training=True):
    if is_training:
        return transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.RandomResizedCrop(image_size, scale=(0.75, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=15),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=cfg.sat_jitter, hue=cfg.hue_jitter),
            transforms.RandomGrayscale(p=cfg.grayscale_prob),
            transforms.RandomSolarize(threshold=128, p=cfg.solarize_prob),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    else:
        return transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

In [ ]:
# 4. Synthetic Mock Dataset Generator (for instant Kaggle/Colab execution)
def generate_mock_sarees(output_dir='./data/sarees', num_designs=8, colorways=6, size=224):
    out_p = Path(output_dir)
    out_p.mkdir(parents=True, exist_ok=True)
    palettes = [
        ((180, 20, 40), (240, 215, 0)),   # Red & Gold
        ((10, 40, 150), (220, 200, 100)), # Royal Blue & Gold
        ((15, 120, 60), (255, 215, 0)),   # Emerald Green & Gold
        ((180, 20, 120), (255, 255, 255)),# Rani Pink & Silver
        ((220, 160, 10), (120, 20, 20)),  # Mustard & Maroon
        ((30, 30, 30), (240, 200, 50)),   # Black & Gold
    ]
    designs = ['peacock', 'temple', 'ikat', 'paisley', 'bandhani', 'floral', 'chevron', 'lattice']
    
    samples = []
    class_to_idx = {d: i for i, d in enumerate(designs[:num_designs])}
    
    for d_idx, d_name in enumerate(designs[:num_designs]):
        f_dir = out_p / f"motif_{d_name}"
        f_dir.mkdir(parents=True, exist_ok=True)
        for c_idx in range(colorways):
            bg, fg = palettes[c_idx % len(palettes)]
            img = Image.new('RGB', (size, size), bg)
            draw = ImageDraw.Draw(img)
            if 'peacock' in d_name:
                for y in range(32, size, 64):
                    for x in range(32, size, 64):
                        draw.ellipse([x-16, y-16, x+16, y+16], fill=fg)
            elif 'temple' in d_name:
                for y in range(0, size, 32):
                    for x in range(0, size, 32):
                        draw.polygon([(x, y+32), (x+16, y), (x+32, y+32)], fill=fg)
            elif 'ikat' in d_name:
                for y in range(0, size, 40):
                    for x in range(0, size, 40):
                        draw.polygon([(x+20, y), (x+40, y+20), (x+20, y+40), (x, y+20)], fill=fg)
            else:
                for y in range(16, size, 32):
                    for x in range(16, size, 32):
                        draw.rectangle([x-6, y-6, x+6, y+6], fill=fg)
                        
            img_p = f_dir / f"colorway_{c_idx+1}.jpg"
            img.save(img_p)
            samples.append((img_p, d_idx))
    return samples, class_to_idx

class SareeDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, label, str(path)

In [ ]:
# 5. Metric Network Architecture & ArcFace Loss
class SareeMetricNet(nn.Module):
    def __init__(self, backbone_name='convnext_tiny', embedding_dim=512):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=True, num_classes=0)
        in_features = self.backbone.num_features
        self.neck = nn.Sequential(
            nn.BatchNorm1d(in_features),
            nn.Dropout(p=0.2),
            nn.Linear(in_features, embedding_dim, bias=False),
            nn.BatchNorm1d(embedding_dim)
        )
    def forward(self, x, normalize=True):
        feat = self.backbone(x)
        emb = self.neck(feat)
        if normalize: emb = F.normalize(emb, p=2, dim=1)
        return emb

class ArcFaceLoss(nn.Module):
    def __init__(self, in_features, num_classes, scale=30.0, margin=0.50):
        super().__init__()
        self.scale = scale
        self.margin = margin
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m = math.cos(margin)
        self.sin_m = math.sin(margin)
        self.th = math.cos(math.pi - margin)
        self.mm = math.sin(math.pi - margin) * margin
        
    def forward(self, embeddings, labels):
        norm_emb = F.normalize(embeddings, p=2, dim=1)
        norm_w = F.normalize(self.weight, p=2, dim=1)
        cosine = torch.clamp(F.linear(norm_emb, norm_w), -1.0 + 1e-7, 1.0 - 1e-7)
        sine = torch.sqrt(1.0 - torch.pow(cosine, 2)).clamp(0.0, 1.0)
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)
        
        one_hot = torch.zeros(cosine.size(), device=embeddings.device)
        one_hot.scatter_(1, labels.view(-1, 1).long(), 1.0)
        logits = ((one_hot * phi) + ((1.0 - one_hot) * cosine)) * self.scale
        return F.cross_entropy(logits, labels)

In [ ]:
# 6. Training Pipeline
samples, class_to_idx = generate_mock_sarees()
num_classes = len(class_to_idx)

# Split into Gallery (50%) and Query (50%)
gal_samples, query_samples = [], []
for d_idx in range(num_classes):
    d_s = [s for s in samples if s[1] == d_idx]
    half = len(d_s) // 2
    gal_samples.extend(d_s[:half])
    query_samples.extend(d_s[half:])

train_loader = DataLoader(SareeDataset(samples, transform=get_transforms(is_training=True)), batch_size=16, shuffle=True)
model = SareeMetricNet(backbone_name=cfg.backbone, embedding_dim=cfg.embedding_dim).to(device)
arcface_head = ArcFaceLoss(cfg.embedding_dim, num_classes, scale=cfg.arcface_scale, margin=cfg.arcface_margin).to(device)
optimizer = torch.optim.AdamW([{'params': model.parameters(), 'lr': cfg.learning_rate}, {'params': arcface_head.parameters(), 'lr': cfg.learning_rate*2}], weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs)

print(f"Starting training for {cfg.epochs} epochs...")
for epoch in range(1, cfg.epochs + 1):
    model.train(); arcface_head.train()
    total_loss = 0.0
    for imgs, lbls, _ in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        embs = model(imgs)
        loss = arcface_head(embs, lbls)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    if epoch % 2 == 0 or epoch == cfg.epochs:
        print(f"Epoch [{epoch:02d}/{cfg.epochs:02d}] Loss: {total_loss/len(train_loader):.4f}")

In [ ]:
# 7. Dual Protocol Evaluation (Identification & Verification)
@torch.no_grad()
def extract_embs(model, loader):
    model.eval()
    embs, lbls, paths = [], [], []
    for img, lbl, p in loader:
        embs.append(model(img.to(device)).cpu().numpy())
        lbls.append(lbl.numpy())
        paths.extend(p)
    return np.concatenate(embs, axis=0), np.concatenate(lbls, axis=0), paths

gal_loader = DataLoader(SareeDataset(gal_samples, transform=get_transforms(is_training=False)), batch_size=16)
qry_loader = DataLoader(SareeDataset(query_samples, transform=get_transforms(is_training=False)), batch_size=16)

gal_e, gal_l, gal_p = extract_embs(model, gal_loader)
qry_e, qry_l, qry_p = extract_embs(model, qry_loader)

# 1:N Identification (Rank-1, Rank-5, mAP)
sim_matrix = np.dot(qry_e, gal_e.T)
rank1_hits, rank5_hits, aps = 0, 0, []
for i in range(len(qry_l)):
    ranked = np.argsort(-sim_matrix[i])
    matches = (gal_l[ranked] == qry_l[i]).astype(int)
    if matches[0]: rank1_hits += 1
    if np.any(matches[:5]): rank5_hits += 1
    cum = np.cumsum(matches)
    prec = cum / np.arange(1, len(matches)+1)
    aps.append(np.sum(prec * matches) / np.sum(matches))

print(f"\n--- IDENTIFICATION RESULTS ---")
print(f"Rank-1 Accuracy : {rank1_hits / len(qry_l) * 100:.2f}%")
print(f"Rank-5 Accuracy : {rank5_hits / len(qry_l) * 100:.2f}%")
print(f"mAP             : {np.mean(aps) * 100:.2f}%")

# 1:1 Verification (ROC-AUC, EER)
all_e = np.concatenate([gal_e, qry_e])
all_l = np.concatenate([gal_l, qry_l])
gen_s, imp_s = [], []
for i in range(len(all_l)):
    for j in range(i+1, len(all_l)):
        s = float(np.dot(all_e[i], all_e[j]))
        if all_l[i] == all_l[j]: gen_s.append(s)
        else: imp_s.append(s)

y_true = [1]*len(gen_s) + [0]*len(imp_s)
y_score = gen_s + imp_s
fpr, tpr, _ = roc_curve(y_true, y_score)
roc_auc = auc(fpr, tpr)
eer = float((fpr[np.nanargmin(np.abs(fpr - (1.0 - tpr)))] + (1.0 - tpr)[np.nanargmin(np.abs(fpr - (1.0 - tpr)))]) / 2.0)

print(f"\n--- VERIFICATION RESULTS ---")
print(f"ROC-AUC : {roc_auc * 100:.2f}%")
print(f"EER     : {eer * 100:.2f}%")

In [ ]:
# 8. Visual Verification Demo (Query vs Gallery Matches)
plt.figure(figsize=(12, 4))
sample_q_idx = 0
q_img = Image.open(qry_p[sample_q_idx])
plt.subplot(1, 4, 1)
plt.imshow(q_img)
plt.title(f"Query (Label: {qry_l[sample_q_idx]})")
plt.axis('off')

sims = np.dot(gal_e, qry_e[sample_q_idx])
top3 = np.argsort(-sims)[:3]
for idx, g_idx in enumerate(top3, start=2):
    g_img = Image.open(gal_p[g_idx])
    plt.subplot(1, 4, idx)
    plt.imshow(g_img)
    plt.title(f"Rank #{idx-1} (Sim: {sims[g_idx]:.3f})")
    plt.axis('off')
plt.tight_layout()
plt.show()